# 1 · Bronze — ingesta con dlt (`bronze_ingest.ipynb`)

Respaldo manual de la tarea **`bronze_ingest`** del DAG `medallon_stackoverflow`. Ejecuta **la misma función** que Airflow (`jobs/bronze_ingest.py`).

- **Fuente:** `votes/2021.parquet` del dataset StackOverflow de ClickHouse (HTTP).
- **Destino:** `s3://bronze/votes/votes_2021/` en MinIO, en **Parquet**.
- **Modo:** `write_disposition="replace"` → **override**: cada ejecución reemplaza el año en Bronze.
- Se conserva el mismo rango de meses que los archivos subidos a mano (`MES_INICIO`–`MES_FIN` en `jobs/comun.py`).

Ejecutar las celdas en orden (*Run All*).

In [1]:
import os, sys
sys.path.insert(0, os.environ.get("JOBS_DIR", "/workspace/jobs"))   # carpeta jobs/ del proyecto
import comun
import bronze_ingest
print("Año del pipeline:", comun.ANIO_PIPELINE, "| meses:", comun.MES_INICIO, "a", comun.MES_FIN)

Año del pipeline: 2021 | meses: 1 a 3


## Ingesta (dlt → MinIO)

In [2]:
bronze_ingest.main()

Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/votes/2021.parquet


Descargado: 165 MB


Filas enviadas a Bronze: 4,831,211


Pipeline bronze_votes load step completed in 0.60 seconds
1 load package(s) were loaded to destination filesystem and into dataset votes
The filesystem destination used s3://bronze location to store data
Load package 1791990992.7422044 is LOADED and contains no failed jobs
Bronze listo: s3://bronze/votes/votes_2021/


## Verificación: contenido de Bronze

In [3]:
import pyarrow.dataset as ds
from pyarrow import fs

access_key, secret_key = comun.credenciales_minio()
s3 = fs.S3FileSystem(access_key=access_key, secret_key=secret_key,
                     endpoint_override="minio:9000", scheme="http", region="us-east-1")

for tabla, anio in [("posts", 2020), ("posts", 2021), ("votes", 2020), ("votes", 2021)]:
    carpeta = comun.ruta_bronze(tabla, anio, esquema="s3").replace("s3://", "")
    archivos = [f.path for f in s3.get_file_info(fs.FileSelector(carpeta)) if f.path.endswith(".parquet")]
    filas = ds.dataset(archivos, filesystem=s3, format="parquet").count_rows()
    print(f"{tabla}_{anio:<6} {len(archivos)} archivo(s)  {filas:>12,} filas   s3://{carpeta}")

posts_2020   1 archivo(s)     1,040,310 filas   s3://bronze/posts/posts_2020/
posts_2021   1 archivo(s)       969,359 filas   s3://bronze/posts/posts_2021/
votes_2020   1 archivo(s)     4,839,670 filas   s3://bronze/votes/votes_2020/
votes_2021   1 archivo(s)     4,831,211 filas   s3://bronze/votes/votes_2021/
